In [1]:
import pyspark

In [2]:
pyspark.__version__

'3.5.1'

In [3]:
pyspark.__file__

'/usr/local/lib/python3.11/dist-packages/pyspark/__init__.py'

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import types

In [5]:
spark = SparkSession.builder \
    .appName("RemoteSparkNotebook") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.547.jar",
        "/opt/spark-extra-jars/hadoop-aws-3.3.6.jar",
        "/opt/spark-extra-jars/hadoop-common-3.3.6.jar",
        "/opt/spark-extra-jars/hadoop-auth-3.3.6.jar"
    ])) \
    .getOrCreate()

25/09/16 09:09:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [6]:
spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"]).show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [7]:
from minio import Minio

In [8]:
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)

In [9]:
bucket = "my-bucket"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [10]:
!mkdir -p data/raw/fhvhv/

In [25]:
!wget https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz -P data/raw/fhvhv/

--2025-09-16 08:57:43--  https://github.com/DataTalksClub/nyc-tlc-data/releases/download/fhvhv/fhvhv_tripdata_2021-01.csv.gz
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/513814948/035746e8-4e24-47e8-a3ce-edcf6d1b11c7?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-09-16T09%3A31%3A16Z&rscd=attachment%3B+filename%3Dfhvhv_tripdata_2021-01.csv.gz&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-09-16T08%3A30%3A39Z&ske=2025-09-16T09%3A31%3A16Z&sks=b&skv=2018-11-09&sig=01X7ww%2BAa%2B9XOIeIw%2BbK33jlVSUZXA1E3aqghBB0lHw%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1ODAxMzM2NCwibmJmIjoxNzU4MDEzMDY0LC

In [11]:
!ls -lh data/raw/fhvhv/

total 129M
-rw-r--r-- 1 jovyan jovyan 124M Jul 14  2022 fhvhv_tripdata_2021-01.csv.gz


In [28]:
file_path = "data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz"  # file inside shared /data
object_name = "data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz"

client.fput_object(bucket, object_name, file_path)

print(f"Uploaded {file_path} → {bucket}/{object_name}")

Uploaded data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz → my-bucket/data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz


In [12]:
# List all buckets
buckets = client.list_buckets()
for bucket in buckets:
    print(bucket.name, bucket.creation_date)

my-bucket 2025-09-16 08:55:49.177000+00:00


In [13]:
bucket = "my-bucket"
for obj in client.list_objects(bucket, recursive=True):
    print(obj.bucket_name, obj.object_name, obj.size)

my-bucket data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz 129967421


In [14]:
schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])

In [15]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("s3a://my-bucket/data/raw/fhvhv/fhvhv_tripdata_2021-01.csv.gz")

25/09/16 09:10:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [16]:
df.show()

[Stage 2:>                                                          (0 + 1) / 1]

+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|hvfhs_license_num|dispatching_base_num|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|SR_Flag|
+-----------------+--------------------+-------------------+-------------------+------------+------------+-------+
|           HV0003|              B02682|2021-01-01 00:33:44|2021-01-01 00:49:07|         230|         166|   NULL|
|           HV0003|              B02682|2021-01-01 00:55:19|2021-01-01 01:18:21|         152|         167|   NULL|
|           HV0003|              B02764|2021-01-01 00:23:56|2021-01-01 00:38:05|         233|         142|   NULL|
|           HV0003|              B02764|2021-01-01 00:42:51|2021-01-01 00:45:50|         142|         143|   NULL|
|           HV0003|              B02764|2021-01-01 00:48:14|2021-01-01 01:08:42|         143|          78|   NULL|
|           HV0005|              B02510|2021-01-01 00:06:59|2021-01-01 00:43:01|

In [17]:
df = df.repartition(24)

In [18]:
df.write.parquet('s3a://my-bucket/data/pq/fhvhv/')

In [19]:
for obj in client.list_objects(bucket, recursive=True):
    print(obj.bucket_name, obj.object_name, obj.size)

my-bucket data/pq/fhvhv/_SUCCESS 0
my-bucket data/pq/fhvhv/part-00000-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9843343
my-bucket data/pq/fhvhv/part-00001-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9838623
my-bucket data/pq/fhvhv/part-00002-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9832345
my-bucket data/pq/fhvhv/part-00003-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9846376
my-bucket data/pq/fhvhv/part-00004-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9832110
my-bucket data/pq/fhvhv/part-00005-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9839000
my-bucket data/pq/fhvhv/part-00006-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9842041
my-bucket data/pq/fhvhv/part-00007-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9841234
my-bucket data/pq/fhvhv/part-00008-b6072928-603c-4c29-b8c9-e1bc8714b9d0-c000.snappy.parquet 9836032
my-bucket data/pq/fhvhv/part-00009-b6072928-603c-4c29-b8c9-e1bc87

In [20]:
df = spark.read.parquet('s3a://my-bucket/data/pq/fhvhv/')

In [21]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- SR_Flag: string (nullable = true)



In [22]:
df.select('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003').show()

+-------------------+-------------------+------------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|
+-------------------+-------------------+------------+------------+
|2021-01-01 16:47:20|2021-01-01 16:58:28|          50|         163|
|2021-01-05 02:00:14|2021-01-05 02:19:39|          48|          95|
|2021-01-02 00:34:43|2021-01-02 00:45:38|          63|          77|
|2021-01-02 16:20:11|2021-01-02 16:56:36|          63|         244|
|2021-01-24 16:00:53|2021-01-24 16:07:40|         210|         165|
|2021-01-16 19:35:17|2021-01-16 19:50:20|         113|         143|
|2021-01-01 11:15:17|2021-01-01 11:24:55|         231|         148|
|2021-01-19 12:05:32|2021-01-19 12:33:46|         228|         210|
|2021-01-17 13:54:52|2021-01-17 14:07:03|          39|          61|
|2021-01-30 18:03:33|2021-01-30 18:23:17|          42|         250|
|2021-01-16 12:36:55|2021-01-16 13:03:23|         131|         265|
|2021-01-30 23:07:14|2021-01-30 23:27:34|       

In [23]:
from pyspark.sql import functions as F

In [24]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-10|  2021-01-10|          97|          25|
| 2021-01-08|  2021-01-08|         138|         265|
| 2021-01-01|  2021-01-01|          50|         163|
| 2021-01-15|  2021-01-15|         163|          79|
| 2021-01-12|  2021-01-12|          47|          74|
| 2021-01-05|  2021-01-05|          48|          95|
| 2021-01-02|  2021-01-02|          63|          77|
| 2021-01-06|  2021-01-06|         238|          41|
| 2021-01-02|  2021-01-02|          63|         244|
| 2021-01-24|  2021-01-24|         210|         165|
| 2021-01-16|  2021-01-16|         113|         143|
| 2021-01-28|  2021-01-28|          91|          89|
| 2021-01-01|  2021-01-01|         231|         148|
| 2021-01-19|  2021-01-19|         228|         210|
| 2021-01-17|  2021-01-17|          39|          61|
| 2021-01-30|  2021-01-30|          42|       

In [25]:
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}'
    elif num % 3 == 0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

In [26]:
print(crazy_stuff('B02682'))

a/a7a


In [27]:
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [28]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()

+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/9ce| 2021-01-10|  2021-01-10|          97|          25|
|  e/9ce| 2021-01-08|  2021-01-08|         138|         265|
|  e/b3c| 2021-01-01|  2021-01-01|          50|         163|
|  e/9ce| 2021-01-15|  2021-01-15|         163|          79|
|  e/9ce| 2021-01-12|  2021-01-12|          47|          74|
|  s/acd| 2021-01-05|  2021-01-05|          48|          95|
|  e/b38| 2021-01-02|  2021-01-02|          63|          77|
|  e/9ce| 2021-01-06|  2021-01-06|         238|          41|
|  e/acc| 2021-01-02|  2021-01-02|          63|         244|
|  e/acc| 2021-01-24|  2021-01-24|         210|         165|
|  e/b35| 2021-01-16|  2021-01-16|         113|         143|
|  e/9ce| 2021-01-28|  2021-01-28|          91|          89|
|  e/acc| 2021-01-01|  2021-01-01|         231|         148|
|  e/b33| 2021-01-19|  2